# Vesicle Detection Training - Clean YOLOv11 Colab Notebook
This notebook prepares and trains a YOLOv11 model using your vesicle dataset from Google Drive.

In [ ]:
# ✅ Step 0: Install environmentA
!pip install -U ultralytics

In [ ]:
# ✅ Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ✅ Step 2: Unzip dataset from Google Drive
# Change this path to match your actual dataset ZIP location
zip_path = '/content/drive/MyDrive/vesicle_dataset.zip'  # ← Update if needed
!unzip -q "$zip_path" -d /content/datasets

In [ ]:
# ✅ Verify dataset structure
!ls /content/datasets   # should say: vesicle_dataset

In [ ]:
# ✅ Step 3: Train YOLOv11 model with periodic saving
import os
os.environ['WANDB_MODE'] = 'disabled'
from ultralytics import YOLO
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Make sure the checkpoints folder exists
!mkdir -p /content/drive/MyDrive/vesicle_project/checkpoints/

import subprocess

# Start background sync of checkpoints to Drive every 60 seconds
sync_script = """
import time
import shutil
import os

src_dir = 'runs/detect/vesicle_project/train/weights'
dst_dir = '/content/drive/MyDrive/vesicle_project/checkpoints'

os.makedirs(dst_dir, exist_ok=True)

while True:
    if os.path.exists(src_dir):
        for filename in os.listdir(src_dir):
            if filename.endswith('.pt'):
                src = os.path.join(src_dir, filename)
                dst = os.path.join(dst_dir, filename)
                try:
                    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
                        shutil.copy2(src, dst)
                except Exception as e:
                    print(f"Error copying {filename}: {e}")
    time.sleep(60)
"""

with open("sync_checkpoints.py", "w") as f:
    f.write(sync_script)

# Start script as background process
process = subprocess.Popen(["python3", "sync_checkpoints.py"])
print("✅ Background sync process started")

# Load pre-trained model
model = YOLO('yolo11m.pt')

# Train model with checkpoint saving every 10 epochs
model.train(
    data='/content/datasets/vesicle_dataset/data.yaml',
    epochs=100, # increase if the model is still improving
    imgsz=1024, #change 1024 to 640 if running out of memory
    batch=8, #change 16 to 8 if running out of memory
    device=0,
    save=True,
    save_period=10,
    project='vesicle_project',
    name='train',
    exist_ok=True,
)

print("✅ Training started with periodic checkpoint saving to Google Drive.")


## ✅ Done. Check `runs/detect/train` for results.
Use the following cell to download the best model weights if needed:

In [ ]:
# ✅ Step 4: Save YOLO training outputs to Google Drive
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define Google Drive save folder
drive_output_folder = '/content/drive/MyDrive/vesicle_project/train'
local_train_folder = 'runs/detect/vesicle_project/train'

# Create output folder on Drive if it doesn't exist
!mkdir -p "$drive_output_folder"

# Safety checks so it doesn't "succeed" while copying nothing
if not os.path.exists(local_train_folder):
    raise FileNotFoundError(
        f"Local training folder not found: {local_train_folder}\n"
        "Training output may be in a different run folder. Try:\n"
        "!find runs -maxdepth 5 -type d -name weights\n"
        "!find runs -name '*.pt'"
    )

# Copy entire train folder to Drive
!cp -a "$local_train_folder"/. "$drive_output_folder"/

# Save the folder path to a reference file
folder_path_file = '/content/drive/MyDrive/vesicle_project/train_folder_path.txt'
with open(folder_path_file, 'w') as f:
    f.write(f"local={local_train_folder}\ndrive={drive_output_folder}\n")

print("✅ Entire YOLO training folder saved to Google Drive at:", drive_output_folder)
print("✅ Folder path saved to train_folder_path.txt")

In [ ]:
# ✅ Step 5: Download trained weights
import os
from google.colab import drive

# Define paths
local_train_folder = '/content/runs/detect/vesicle_project/train'
drive_train_folder = '/content/drive/MyDrive/vesicle_project/train'
best_pt_path = os.path.join(local_train_folder, 'weights', 'best.pt')
results_png_path = os.path.join(local_train_folder, 'results.png')

# Mount Google Drive
drive.mount('/content/drive')

# Check if key files are missing
needs_restore = (
    not os.path.exists(local_train_folder) or
    not os.path.exists(best_pt_path) or
    not os.path.exists(results_png_path)
)

if needs_restore:
    print('⚠ Training folder incomplete or missing in Colab. Restoring from Google Drive...')
    os.makedirs(local_train_folder, exist_ok=True)
    !cp -a "$drive_train_folder"/. "$local_train_folder"/
    print('✅ Training folder restored to Colab.')
else:
    print('✅ Training folder and key files already present in Colab.')

# Verify important files
if os.path.exists(best_pt_path):
    print('✅ best.pt is available at:', best_pt_path)
else:
    print('⚠ Warning: best.pt not found inside the restored folder.')

if os.path.exists(results_png_path):
    print('✅ results.png is available at:', results_png_path)
else:
    print('⚠ Warning: results.png not found inside the restored folder.')

# List final contents
print('\n📦 Final folder contents:')
!ls -lh "$local_train_folder"

In [ ]:

# 📉 Step 6: Visualize training curves (metrics)
from IPython.display import Image, display
metrics_img_path = 'runs/detect/vesicle_project/train/results.png'
display(Image(filename=metrics_img_path))


In [ ]:
# ✅ Step 7: Check and unzip dataset from Google Drive if needed

import os

dataset_root = '/content/datasets'
dataset_images_val = '/content/datasets/images/val'
dataset_labels_val = '/content/datasets/labels/val'

# Check if dataset is complete
dataset_ok = (
    os.path.exists(dataset_images_val) and
    os.path.exists(dataset_labels_val) and
    len(os.listdir(dataset_images_val)) > 0 and
    len(os.listdir(dataset_labels_val)) > 0
)

if dataset_ok:
    print("✅ Dataset already exists and is complete. Skipping unzip.")
else:
    print("⏳ Dataset missing or incomplete. Extracting from Google Drive...")
    !unzip -o /content/drive/MyDrive/vesicle_dataset.zip -d "$dataset_root"
    print("✅ Dataset extracted from Google Drive")

In [ ]:
# ✅ Step 8: Restore YOLO training outputs from Google Drive

!mkdir -p /content/runs/detect/vesicle_project
!cp -a /content/drive/MyDrive/vesicle_project/train/. /content/runs/detect/vesicle_project/train/

print("✅ YOLO outputs restored from Google Drive")

In [ ]:

# 🔍 Step 9: Show example predictions
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import random
import os

model = YOLO('runs/detect/vesicle_project/train/weights/best.pt')
image_dir = '/content/datasets/vesicle_dataset/images/val'
images = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]
sample_images = random.sample(images, min(10, len(images)))

for i, img_path in enumerate(sample_images):
    results = model(img_path, conf=0.05)  # Lower threshold → more detections
    results[0].save(filename=f"prediction_{i}.jpg")
    display(Image.open(f"prediction_{i}.jpg"))


In [ ]:

# 💾 Step 10: Zip results and generate download link
import shutil
from google.colab import files

shutil.make_archive("yolo_results", 'zip', "runs/detect/vesicle_project/train")
files.download("yolo_results.zip")


In [ ]:
# Step 11: FINAL PREDICTION ON ALL IMAGES

# Predict on validation images
model.predict('/content/datasets/vesicle_dataset/images/val', save=True, save_txt=True, conf=0.1)
!mv runs/detect/predict runs/detect/predict_val

# Predict on training images
model.predict('/content/datasets/vesicle_dataset/images/train', save=True, save_txt=True, conf=0.1)
!mv runs/detect/predict runs/detect/predict_train

# ✅ FLATTEN FOLDERS

import os
import shutil

# Create flat folders
os.makedirs('vesicle_predictions/images', exist_ok=True)
os.makedirs('vesicle_predictions/labels', exist_ok=True)

# Move all images and labels into flat folders
def move_files(src_folder):
    for item in os.listdir(src_folder):
        item_path = os.path.join(src_folder, item)
        if os.path.isfile(item_path) and item.endswith('.jpg'):
            shutil.move(item_path, f'vesicle_predictions/images/{item}')
        elif item == 'labels':
            label_path = os.path.join(src_folder, 'labels')
            for label_file in os.listdir(label_path):
                shutil.move(os.path.join(label_path, label_file), f'vesicle_predictions/labels/{label_file}')

# Apply to both val and train predictions
move_files('runs/detect/predict_val')
move_files('runs/detect/predict_train')

# ✅ Remove old prediction folders
shutil.rmtree('runs/detect/predict_val')
shutil.rmtree('runs/detect/predict_train')

# ✅ Remove entire runs folder if it exists
if os.path.exists('runs'):
    shutil.rmtree('runs')

# ✅ ZIP AND DOWNLOAD

# Zip only the clean vesicle_predictions folder
!zip -r vesicle_predictions.zip vesicle_predictions

# Download the zip file
from google.colab import files
files.download('vesicle_predictions.zip')